# CoupledCF on MovieLens 1M

This notebook checks the complete data, exercises one real optimizer step, reads the completed experiment comparison, and recomputes a retained checkpoint. Full training is an explicit command in a separate cell. See the report for partial-reproduction boundaries. Fill names and actual contributions in `configs/student.json`.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / 'src/coupledcf').exists():
    ROOT = ROOT.parent
assert (ROOT / 'configs/main.json').exists(), 'Run inside the project or its notebooks directory'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import torch
from coupledcf.data import DataBundle, training_samples
from coupledcf.models import ModelConfig, build_model
from coupledcf.engine import set_seed, evaluate
print('Python', sys.version.split()[0], 'PyTorch', torch.__version__, 'CUDA', torch.cuda.is_available())

## Data and leakage checks

All recorded ratings are positives. The fixed split contains one validation target and one test target per user. Each candidate row contains one target and 99 unobserved items. The complete observed mask prevents held-out positives from becoming negatives.

In [ ]:
data = DataBundle.load(ROOT / 'data/processed/split.npz')
print(data.validate())
manifest = json.loads((ROOT / 'data/processed/manifest.json').read_text())
print({key: manifest[key] for key in ['ratings','rated_items','metadata_items','split_seed','split_sha256']})

## Directly implemented coupling mechanism

The explicit outer product is 8 by 8. A valid 3 by 3 CNN, eight channels and 2 by 2 max pool produce 72 local values. The flattened original matrix contributes 64 global values. DeepCF uses an elementwise product of 32-dimensional ID vectors. The following bounded update is an execution check, separate from the retained scientific results.

In [ ]:
set_seed(123, 2)
smoke_model = build_model(data, ModelConfig(variant='coupled'))
optimizer = torch.optim.Adam(smoke_model.parameters(), lr=.001)
u, i, y = training_samples(data, seed=123)
u, i, y = [torch.from_numpy(a[:4096]) for a in (u, i, y)]
smoke_model.train()
optimizer.zero_grad()
loss = torch.nn.functional.binary_cross_entropy_with_logits(smoke_model(u, i), y)
loss.backward()
assert smoke_model.local_conv.weight.grad.abs().sum() > 0
assert smoke_model.user_id_embedding.weight.grad.abs().sum() > 0
optimizer.step()
print('One completed optimizer update, BCE', float(loss.detach()))
del smoke_model, optimizer

## Completed model comparison

These tables load the saved measurements. Three seeds share one split, not three independently sampled datasets. Error bars show sample standard deviation. The plot and table are generated by the same result summarizer.

In [ ]:
summary = pd.read_csv(ROOT / 'results/summary.csv')
display(summary[['variant','runs','HR@10_mean','HR@10_std','NDCG@10_mean','NDCG@10_std','parameters']])
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'results/figures/model_comparison.png')))

In [ ]:
ablation = pd.read_csv(ROOT / 'results/ablation.csv')
display(ablation.groupby('comparison')[['delta_HR@10','delta_NDCG@10']].agg(['mean','std']))
display(Image(filename=str(ROOT / 'results/figures/learning_curves.png')))

## Recompute a selected checkpoint

The checkpoint was chosen by validation NDCG@10, before the reported test evaluation. Reload it and confirm exact metric agreement on this environment.

In [ ]:
run = ROOT / 'results/main/coupled_seed42'
record = json.loads((run / 'result.json').read_text())
model = build_model(data, ModelConfig(**record['config']['model']))
checkpoint = torch.load(run / 'best.pt', map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint['state_dict'])
metrics, ranks = evaluate(model, data.test, 'cpu')
assert metrics == record['test']
print('PASS selected epoch', record['selected_epoch'], metrics)

## Full reproduction commands

Run these commands in a project terminal. To retrain completed experiments, copy the main config and change `output_dir`. The default preserves matching completed runs.

```powershell
.venv\Scripts\python.exe scripts\run_experiments.py --config configs\main.json
.venv\Scripts\python.exe scripts\summarize_results.py
.venv\Scripts\python.exe scripts\verify_submission.py
```

The study uses a bounded CPU training budget, added validation holdout and disclosed side-feature projections. Sampled warm-user ranking does not establish cold-start quality, full-catalogue retrieval quality or cross-dataset generalization. The field-aware attention proposal in the report is a hypothesis, not a measured improvement.